# Junior results

Imitation pretraining of the GNN (GAT) policy on the four Tutor datasets (hard labels, soft labels
with τ ∈ {0.005, 0.01, 0.02}). Per dataset, an Optuna study of 40 trials
(`experiments/pretrain_hpo.py`) minimises the validation **KL = CE − H(target)**, the part of the
cross-entropy the network can still reduce. The best trial per dataset is then rolled out greedily
on the 144 chronics of `l2rpn_wcci_2020_val`, next to the do-nothing agent on the same chronics and
seeds.

Expected local layout (rsync from the cluster):

```
results/pretraining/junior_<dataset>/   study.log, trials.csv, best -> trial_<n>/ (history.csv,
                                        test_metrics.json, rollout/episodes_valenv_*.csv)
results/pretraining/junior_do_nothing/  rollout/episodes_valenv_*.csv
results/tutor/summary/episodes.csv      Tutor run (optional, for dataset rows per month)
```

In [ ]:
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import optuna
import pandas as pd
from matplotlib.lines import Line2D
from matplotlib.ticker import MaxNLocator

ROOT = Path("..").resolve()
sys.path.insert(0, str(ROOT / "src"))

from analysis.episodes import load_episodes
from imitation.hpo import BEST_LINK, TRIALS_FILE, open_study

optuna.logging.set_verbosity(optuna.logging.WARNING)

# ── Paths ──
RESULTS = ROOT / "results/pretraining"
STUDIES = {  # label -> study directory (experiment.name of experiments/pretrain_hpo.py)
    "hard": RESULTS / "junior_hard",
    "soft τ=0.005": RESULTS / "junior_soft_t0.005",
    "soft τ=0.01": RESULTS / "junior_soft_t0.01",
    "soft τ=0.02": RESULTS / "junior_soft_t0.02",
}
DO_NOTHING = RESULTS / "junior_do_nothing"
TUTOR_EPISODES = ROOT / "results/tutor/summary/episodes.csv"
FIGURES = ROOT / "experiments/figures/junior"
FIGURES.mkdir(parents=True, exist_ok=True)

# ── Constants ──
STEPS_PER_DAY = 288  # 5-minute intervals
MONTHS = ["january", "february", "march", "april", "may", "june",
          "july", "august", "september", "october", "november", "december"]
MONTH_LABELS = [m[:3].capitalize() for m in MONTHS]
DN_LABEL = "do-nothing"

# ── Palette: one fixed categorical slot per dataset, do-nothing as neutral reference ──
DATASET_COLORS = dict(zip(STUDIES, ["#2a78d6", "#eb6834", "#1baf7a", "#eda100"]))
AGENT_COLORS = {**DATASET_COLORS, DN_LABEL: "#a3a29c"}
SURFACE, INK, INK_MUTED, GRID = "#fcfcfb", "#0b0b0b", "#52514e", "#e4e3df"


def style(ax: plt.Axes) -> None:
    ax.set_facecolor(SURFACE)
    for side in ("top", "right", "left"):
        ax.spines[side].set_visible(False)
    ax.spines["bottom"].set_color(GRID)
    ax.tick_params(colors=INK_MUTED, length=0)
    ax.yaxis.grid(True, color=GRID, linewidth=0.8)
    ax.set_axisbelow(True)


def save(fig: plt.Figure, name: str) -> None:
    """Write the figure to experiments/figures/junior/<name>.png (for the Obsidian note)."""
    fig.savefig(FIGURES / f"{name}.png", dpi=150, facecolor=SURFACE, bbox_inches="tight")


def add_month(df: pd.DataFrame) -> pd.DataFrame:
    df["month"] = pd.Categorical(df["chronic"].str.split("_").str[1], categories=MONTHS, ordered=True)
    return df

In [ ]:
studies = {label: open_study(path, path.name) for label, path in STUDIES.items()}
trials = pd.concat([pd.read_csv(path / TRIALS_FILE).assign(dataset=label) for label, path in STUDIES.items()],
                   ignore_index=True)
best_dirs = {label: (path / BEST_LINK).resolve() for label, path in STUDIES.items()}
histories = {label: pd.read_csv(d / "history.csv") for label, d in best_dirs.items()}
test_metrics = pd.DataFrame({label: json.loads((d / "test_metrics.json").read_text())
                             for label, d in best_dirs.items()}).T

episodes = pd.concat(
    [load_episodes(*sorted((d / "rollout").glob("episodes_valenv_*.csv"))).assign(agent=label)
     for label, d in best_dirs.items()]
    + [load_episodes(*sorted((DO_NOTHING / "rollout").glob("episodes_valenv_*.csv"))).assign(agent=DN_LABEL)],
    ignore_index=True)
episodes = add_month(episodes)
episodes["days_survived"] = episodes["steps_survived"] / STEPS_PER_DAY
AGENTS = list(STUDIES) + [DN_LABEL]

print(trials.groupby(["dataset", "state"], sort=False).size().unstack(fill_value=0))
print(episodes.groupby("agent", sort=False)["chronic"].nunique().rename("val chronics rolled out"))

# Optimization history

Every trial's validation KL (filled: complete, hollow: pruned at its last reported epoch) and the
best complete value so far. KL is comparable between datasets in units (nats), but each study fits
its own targets.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(11, 7), facecolor=SURFACE)
for ax, (label, group) in zip(axes.flat, trials.groupby("dataset", sort=False)):
    color = DATASET_COLORS[label]
    group = group.sort_values("number")
    done, pruned = group[group["state"] == "COMPLETE"], group[group["state"] == "PRUNED"]
    ax.scatter(pruned["number"], pruned["value"], s=40, facecolors="none", edgecolors=color, linewidths=1.2)
    ax.scatter(done["number"], done["value"], s=40, color=color, edgecolors=SURFACE, linewidths=1.5)
    best = done.set_index("number")["value"].cummin().reindex(group["number"]).ffill()
    ax.plot(best.index, best.values, color=INK_MUTED, linewidth=2, drawstyle="steps-post")
    ax.set_title(f"{label}  (best {done['value'].min():.3f})", loc="left", color=INK, fontsize=11)
    ax.set_xlabel("Trial", color=INK_MUTED)
    ax.xaxis.set_major_locator(MaxNLocator(integer=True))
    ax.set_ylabel("Validation KL [nats]", color=INK_MUTED)
    style(ax)
handles = [Line2D([0], [0], marker="o", color="none", markerfacecolor=INK_MUTED, markeredgecolor=INK_MUTED, label="complete"),
           Line2D([0], [0], marker="o", color="none", markerfacecolor="none", markeredgecolor=INK_MUTED, label="pruned"),
           Line2D([0], [0], color=INK_MUTED, linewidth=2, label="best so far")]
fig.legend(handles=handles, frameon=False, ncols=3, loc="upper right", labelcolor=INK)
fig.suptitle("Junior HPO: validation KL per trial", x=0.01, ha="left", color=INK, fontsize=12)
fig.tight_layout(rect=(0, 0, 1, 0.95))
save(fig, "optimization_history")
plt.show()

# Hyperparameter importance

fANOVA importance (`optuna.importance.get_param_importances`) of each hyperparameter for the
validation KL, per study and over complete trials. Importances sum to 1 within a study.

In [ ]:
importance = pd.DataFrame({label: optuna.importance.get_param_importances(study)
                           for label, study in studies.items()}).fillna(0.0)
importance.index = [name.split(".", 1)[-1] for name in importance.index]
importance = importance.loc[importance.mean(axis=1).sort_values().index]

fig, ax = plt.subplots(figsize=(10, 6), facecolor=SURFACE)
height = 0.8 / len(STUDIES)
y = np.arange(len(importance))
for i, label in enumerate(STUDIES):
    ax.barh(y - (i - (len(STUDIES) - 1) / 2) * height, importance[label], height,
            color=DATASET_COLORS[label], edgecolor=SURFACE, linewidth=1, label=label)
ax.set_yticks(y, importance.index)
ax.set_xlabel("Importance (fANOVA)", color=INK_MUTED)
ax.set_title("Hyperparameter importance for validation KL", loc="left", color=INK, fontsize=12)
ax.legend(frameon=False, labelcolor=INK, loc="lower right")
style(ax)
ax.yaxis.grid(False)
ax.xaxis.grid(True, color=GRID, linewidth=0.8)
fig.tight_layout()
save(fig, "param_importance")
plt.show()

importance.iloc[::-1].round(3)

# Best trial per dataset

Hyperparameters of the best trial and its metrics at the best epoch. `val_*`: validation split
(model selection), `test_*`: test split (reported only). Accuracy, top5, good action rate, regret
and do-nothing rate are measured against the Tutor's simulated max rho and executed action, so they
are comparable across datasets (the argmax of every soft label is the Tutor's action).

In [ ]:
params = [c for c in trials.columns if c.startswith("params_")]
best_rows = (trials[trials["state"] == "COMPLETE"].sort_values("value")
             .groupby("dataset", sort=False).head(1).set_index("dataset").loc[list(STUDIES)])
best_table = best_rows[["number", "user_attrs_best_epoch", "user_attrs_n_epochs"] + params].rename(
    columns=lambda c: c.replace("params_", "").replace("user_attrs_", ""))
best_table

In [ ]:
METRIC_COLUMNS = ["loss", "entropy", "kl", "accuracy", "top5", "good_action_rate", "regret", "invalid_rate", "do_nothing_rate"]
val = best_rows[[f"user_attrs_val_{m}" for m in METRIC_COLUMNS]].set_axis(METRIC_COLUMNS, axis=1)
summary = pd.concat({"val": val, "test": test_metrics[METRIC_COLUMNS].astype(float)}, axis=1)
summary.round(4)

# Learning curves of the best trials

Train and validation cross-entropy per epoch. The dashed line is the mean entropy of the validation
targets, the lower bound of the validation CE (0 for hard labels); the gap to it is the KL.

In [ ]:
fig, axes = plt.subplots(1, len(STUDIES), figsize=(14, 3.8), facecolor=SURFACE, sharey=False)
for ax, label in zip(axes, STUDIES):
    h, color = histories[label], DATASET_COLORS[label]
    ax.plot(h["epoch"], h["train_loss"], color=color, linewidth=2, linestyle=":", label="train CE")
    ax.plot(h["epoch"], h["val_loss"], color=color, linewidth=2, label="val CE")
    ax.axhline(h["val_entropy"].iloc[0], color=INK_MUTED, linewidth=1, linestyle="--", label="val target entropy")
    best = h.loc[h["val_kl"].idxmin()]
    ax.scatter([best["epoch"]], [best["val_loss"]], s=50, color=color, edgecolors=SURFACE, linewidths=2, zorder=3)
    ax.set_title(label, loc="left", color=INK, fontsize=11)
    ax.set_xlabel("Epoch", color=INK_MUTED)
    ax.xaxis.set_major_locator(MaxNLocator(integer=True))
    style(ax)
axes[0].set_ylabel("Cross-entropy [nats]", color=INK_MUTED)
handles = [Line2D([0], [0], color=INK_MUTED, linewidth=2, linestyle=":", label="train CE"),
           Line2D([0], [0], color=INK_MUTED, linewidth=2, label="val CE"),
           Line2D([0], [0], color=INK_MUTED, linewidth=1, linestyle="--", label="val target entropy"),
           Line2D([0], [0], marker="o", color="none", markerfacecolor=INK_MUTED, label="best epoch")]
fig.legend(handles=handles, frameon=False, ncols=4, loc="upper right", labelcolor=INK)
fig.suptitle("Best trials: learning curves", x=0.01, ha="left", color=INK, fontsize=12)
fig.tight_layout(rect=(0, 0, 1, 0.9))
save(fig, "learning_curves")
plt.show()

# Decision quality on the test split

Rates over the test states of the Tutor's dataset: argmax equals the Tutor's action (accuracy),
the Tutor's action is among the 5 highest logits (top5), argmax within 0.01 rho of the best valid
action (good action rate), argmax is do-nothing (the Tutor executed do-nothing in 3.6 % of states).

In [ ]:
RATES = {"accuracy": "accuracy", "top5": "top-5 accuracy", "good_action_rate": "good action rate",
         "do_nothing_rate": "do-nothing rate"}
fig, ax = plt.subplots(figsize=(10, 4.5), facecolor=SURFACE)
width = 0.8 / len(STUDIES)
x = np.arange(len(RATES))
for i, label in enumerate(STUDIES):
    bars = ax.bar(x + (i - (len(STUDIES) - 1) / 2) * width, test_metrics.loc[label, list(RATES)].astype(float),
                  width, color=DATASET_COLORS[label], edgecolor=SURFACE, linewidth=2, label=label)
    ax.bar_label(bars, fmt="%.2f", fontsize=7, color=INK_MUTED)
ax.set_xticks(x, RATES.values())
ax.set_ylim(0, 1.08)
ax.set_ylabel("Share of test states", color=INK_MUTED)
ax.set_title("Best trials: decision quality on the test split", loc="left", color=INK, fontsize=12)
ax.legend(frameon=False, ncols=len(STUDIES), loc="upper right", bbox_to_anchor=(1, 1.12), labelcolor=INK)
style(ax)
fig.tight_layout()
save(fig, "test_decision_quality")
plt.show()

# Survival on the validation env

Greedy rollouts of the best trials on all chronics of `l2rpn_wcci_2020_val` (never used by Teacher,
Tutor or pretraining), and do-nothing on the same chronics with the same seeds (same maintenance).
Both ratios per agent: survived chronics / chronics and survived steps / total steps.

In [ ]:
def survival_ratios(df: pd.DataFrame, by: list[str]) -> pd.DataFrame:
    """Chronics, survived chronics and both survival ratios per group."""
    out = df.groupby(by, observed=False, sort=False).agg(
        chronics=("chronic", "size"), survived_chronics=("survived", "sum"),
        steps_survived=("steps_survived", "sum"), steps_total=("max_step", "sum"))
    out["chronic_ratio"] = out["survived_chronics"] / out["chronics"]
    out["step_ratio"] = out["steps_survived"] / out["steps_total"]
    return out


overall = survival_ratios(episodes, ["agent"]).loc[AGENTS]
overall.round(3)

In [ ]:
per_month = survival_ratios(episodes, ["agent", "month"])
labels = MONTH_LABELS + ["Total"]


def draw_month_bars(ratio: str, ylabel: str, title: str, name: str) -> None:
    """Grouped bars per month (+ Total) with one bar per agent."""
    fig, ax = plt.subplots(figsize=(14, 4.5), facecolor=SURFACE)
    width = 0.8 / len(AGENTS)
    x = np.arange(len(labels))
    for i, agent in enumerate(AGENTS):
        values = list(per_month.loc[agent, ratio].reindex(MONTHS)) + [overall.loc[agent, ratio]]
        ax.bar(x + (i - (len(AGENTS) - 1) / 2) * width, values, width,
               color=AGENT_COLORS[agent], edgecolor=SURFACE, linewidth=1, label=agent)
    ax.axvline(x=len(MONTH_LABELS) - 0.5, color=GRID, linewidth=1, linestyle="--")
    counts = episodes[episodes["agent"] == AGENTS[0]].groupby("month", observed=False).size()
    ax.set_xticks(x, [f"{m}\n(n={counts[month]})" for m, month in zip(MONTH_LABELS, MONTHS)]
                  + [f"Total\n(n={int(overall.iloc[0]['chronics'])})"])
    ax.set_ylim(0, 1.08)
    ax.set_ylabel(ylabel, color=INK_MUTED)
    ax.set_title(title, loc="left", color=INK, fontsize=12)
    ax.legend(frameon=False, ncols=len(AGENTS), loc="upper right", bbox_to_anchor=(1, 1.14), labelcolor=INK)
    style(ax)
    fig.tight_layout()
    save(fig, name)
    plt.show()


draw_month_bars("step_ratio", "Survived steps / total steps", "Val env: survived steps per month", "val_step_ratio_per_month")
draw_month_bars("chronic_ratio", "Survived chronics / chronics", "Val env: fully survived chronics per month",
                "val_chronic_ratio_per_month")
per_month[["chronic_ratio", "step_ratio"]].unstack("agent").round(2)

# Gain over do-nothing per chronic

Days survived by the junior minus days survived by do-nothing on the same chronic (same seed).
Positive: the policy's topology actions extended survival; negative: they shortened it.

In [ ]:
dn = episodes[episodes["agent"] == DN_LABEL].set_index("chronic")["days_survived"]
gain = {label: (episodes[episodes["agent"] == label].set_index("chronic")["days_survived"] - dn).dropna()
        for label in STUDIES}

fig, ax = plt.subplots(figsize=(10, 4.5), facecolor=SURFACE)
boxes = ax.boxplot(list(gain.values()), tick_labels=list(gain), showmeans=True, patch_artist=True,
                   whiskerprops=dict(color=INK_MUTED), capprops=dict(color=INK_MUTED),
                   medianprops=dict(color=INK, linewidth=1.5),
                   flierprops=dict(marker="o", markeredgecolor=INK_MUTED, markersize=3),
                   meanprops=dict(marker="^", markerfacecolor="white", markeredgecolor=INK))
for patch, label in zip(boxes["boxes"], gain):
    patch.set(facecolor=DATASET_COLORS[label], edgecolor=DATASET_COLORS[label], alpha=0.8)
ax.axhline(0, color=INK_MUTED, linewidth=1, linestyle="--")
ax.set_ylabel("Days survived − do-nothing [days]", color=INK_MUTED)
ax.set_title("Val env: survival gain over do-nothing per chronic", loc="left", color=INK, fontsize=12)
style(ax)
fig.tight_layout()
save(fig, "val_gain_over_do_nothing")
plt.show()

pd.DataFrame({label: {"better": (g > 0).mean(), "equal": (g == 0).mean(), "worse": (g < 0).mean(),
                      "mean gain [days]": g.mean()} for label, g in gain.items()}).T.round(3)

# Dataset coverage vs. survival

Does a month's share of the imitation data relate to the junior's survival in that month on the
val env? x: Tutor dataset rows of the month (Tutor run on the train env), y: survived-step ratio
of the junior minus that of do-nothing in the month (the gain the policy adds). Needs the Tutor
summary (`TUTOR_EPISODES`).

In [ ]:
if TUTOR_EPISODES.exists():
    tutor = add_month(pd.read_csv(TUTOR_EPISODES))
    rows_per_month = tutor.groupby("month", observed=False)["n_decisions"].sum()
    dn_ratio = per_month.loc[DN_LABEL, "step_ratio"]
    fig, axes = plt.subplots(1, len(STUDIES), figsize=(14, 3.8), facecolor=SURFACE, sharey=True)
    for ax, label in zip(axes, STUDIES):
        gain_ratio = (per_month.loc[label, "step_ratio"] - dn_ratio).reindex(MONTHS)
        ax.scatter(rows_per_month.reindex(MONTHS), gain_ratio, s=50, color=DATASET_COLORS[label],
                   edgecolors=SURFACE, linewidths=2)
        for month, short in zip(MONTHS, MONTH_LABELS):
            ax.annotate(short, (rows_per_month[month], gain_ratio[month]), xytext=(4, 3),
                        textcoords="offset points", fontsize=7, color=INK_MUTED)
        ax.axhline(0, color=INK_MUTED, linewidth=1, linestyle="--")
        ax.set_title(label, loc="left", color=INK, fontsize=11)
        ax.set_xlabel("Tutor dataset rows", color=INK_MUTED)
        style(ax)
    axes[0].set_ylabel("Step ratio − do-nothing", color=INK_MUTED)
    fig.suptitle("Dataset rows per month vs. survival gain on the val env", x=0.01, ha="left", color=INK, fontsize=12)
    fig.tight_layout(rect=(0, 0, 1, 0.92))
    save(fig, "coverage_vs_gain")
    plt.show()
else:
    print(f"{TUTOR_EPISODES} not found: sync the Tutor summary to draw this plot")